# The Agent Loop
### Pune AI Builders Meetup — Notebook 04

---

> **Where we left off:** We can call tools. Ask a question → LLM uses tools → returns an answer.
> But WE still decide what to ask. WE decide when to stop.
>
> **New question:** *What if the software drove the session — not the human?*

That's an **Agent**.

---
## The Problem with Notebook 03

```
NB03:   You ask a question
           → LLM uses tools
           → Returns answer
           → You ask the next question   ← YOU are still in the loop
```

A real career counsellor doesn't wait to be asked:

- *"Let me check your GitHub before we discuss your experience."*
- *"I found 3 strong matches — here's why each fits your goals."*
- *"Two skill gaps to close before you apply."*

The counsellor **drives the session**. The candidate responds.

---
## Chatbot vs Tool Caller vs Agent

| | Chatbot | Tool Caller (NB03) | **Agent (NB04)** |
|---|---|---|---|
| Driven by | Human questions | Human questions | **Its own goal** |
| Uses tools? | No | Yes, when asked | **Yes, autonomously** |
| Tracks progress? | No | No | **Yes** |
| Knows when done? | No | No | **Yes** |

```
Agent = Goal  →  Decide  →  Act (tools)  →  Observe  →  repeat until Goal met
```

In [6]:
import os, json, requests
import pandas as pd
from openai import OpenAI

OPENAI_API_KEY = "your-api-key-here"  # ← replace with your key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY))

# Complete candidate profile — result of NB01 + NB02
ARJUN = {
    "name"              : "Arjun Mehta",
    "current_role"      : "Senior SDE-II at Flipkart",
    "total_experience"  : 8,
    "top_skills"        : ["Java", "Go", "Kafka", "Redis", "distributed systems"],
    "career_goal"       : "Senior IC or Tech Lead in fintech/infra",
    "salary_expectation": "42–50 LPA",
    "notice_period"     : "60 days",
    "reason_for_change" : "Work became predictable. Wants harder problems.",
    "gap_explanation"   : "First gap personal. Second intentional — built review-gpt.",
    "open_to_relocation": "Bangalore preferred, open to remote",
    "github"            : "arjun91",
}

JOB_DATABASE = pd.read_csv("../data/jobs.csv")
print("Setup done.")

Setup done.


---
## Step 1 — Goal vs Role

This is the most important change.

**NB03 system prompt** (a *role*):
```
"You are a career counsellor."
```

**NB04 system prompt** (a *goal*):
```
"Your goal is to produce a complete career assessment.
 Step 1: verify GitHub. Step 2: find matching jobs. Step 3: write recommendation.
 Work autonomously. Call the finish_session tool when done."
```

A role tells the LLM *what it is*. A goal tells the LLM *what to accomplish*.
The goal is what transforms a chatbot into an agent.

In [8]:
AGENT_GOAL = """
You are an AI career counsellor agent. Your goal is to produce a complete career assessment.

Work through these steps autonomously — do not ask the user what to do next:
  1. Verify the candidate's GitHub to validate their technical claims.
  2. Search for the 3 best matching jobs based on their skills and experience.
  3. Call finish_session with your final recommendation and next steps.

Be specific. Reference actual repos, actual job titles, actual skill gaps.
if githu fetch fails ignore and continue to next step. If no jobs found, return a message saying so.
dont fake any jobs or dont look jobs from your own knowledge. Only use the jobs in the JOB_DATABASE.
Call finish_session when you have completed all three steps.
"""

print("Agent goal defined.")

Agent goal defined.


---
## Step 2 — Add a `finish_session` Tool

In NB03, the loop stopped when the LLM had no more tool calls.
But that's implicit — we can't tell *why* it stopped.

In an agent, we give the LLM an **explicit exit**: a `finish_session` tool.
When the agent calls it, the session ends — and we get structured output.

```
Agent calls finish_session(recommendation, top_jobs, next_steps)
                                ↑
              This is the agent saying: "I'm done. Here's my output."
```

In [3]:
# ── Tool implementations ───────────────────────────────────────────────────

def read_github(username: str) -> dict:
    """Fetch real GitHub profile via public API. Gracefully handles rate-limits."""
    demo_username = "mahadevTW"
    try:
        user  = requests.get(f"https://api.github.com/users/{demo_username}", timeout=5).json()
        repos = requests.get(f"https://api.github.com/users/{demo_username}/repos?sort=stars&per_page=10", timeout=5).json()
    except Exception as e:
        return {"error": str(e)}

    if not isinstance(user, dict) or "login" not in user:
        return {"error": user.get("message", "GitHub API unavailable") if isinstance(user, dict) else "GitHub API unavailable"}
    if not isinstance(repos, list):
        return {"error": repos.get("message", "Could not fetch repos") if isinstance(repos, dict) else "Could not fetch repos"}

    lang_count = {}
    for r in repos:
        if r.get("language"):
            lang_count[r["language"]] = lang_count.get(r["language"], 0) + 1
    return {
        "username"     : demo_username,
        "name"         : user.get("name"),
        "company"      : user.get("company"),
        "public_repos" : user.get("public_repos"),
        "followers"    : user.get("followers"),
        "top_languages": sorted(lang_count, key=lang_count.get, reverse=True)[:3],
        "top_repos"    : [{"name": r["name"], "stars": r["stargazers_count"],
                           "lang": r["language"], "desc": r["description"],
                           "last_pushed": r["pushed_at"][:10]} for r in repos[:5]],
    }

def search_jobs(skills: list, domain: str = None, min_years: int = 0) -> list:
    """Search job database by matching candidate skills against job descriptions."""
    results = []
    active_jobs = JOB_DATABASE[JOB_DATABASE["is_active"] == True]

    for _, row in active_jobs.iterrows():
        searchable = " ".join([
            str(row.get("job_title", "") or ""),
            str(row.get("job_description", "") or ""),
        ]).lower()

        matched = [s for s in skills if s.strip().lower() in searchable]
        overlap  = len(matched) / len(skills) if skills else 0

        if overlap >= 0.4:
            results.append({
                "id"            : row["id"],
                "job_uid"       : row["job_uid"],
                "job_title"     : row["job_title"],
                "company"       : row["company_name"],
                "location"      : row["location"],
                "salary"        : row.get("salary") or "Not disclosed",
                "matched_skills": matched,
                "match_score"   : round(overlap * 100),
                "apply_link"    : row["apply_link"],
                "description"   : str(row.get("job_description", ""))[:300] + "...",
            })

    return sorted(results, key=lambda x: x["match_score"], reverse=True)[:3]

def finish_session(recommendation: str, top_jobs: list, next_steps: list) -> dict:
    """Agent calls this when the assessment is complete."""
    return {"status": "completed", "recommendation": recommendation,
            "top_jobs": top_jobs, "next_steps": next_steps}


# ── Tool dispatcher ────────────────────────────────────────────────────────

def execute_tool(name: str, args: dict):
    if name == "read_github"   : return read_github(**args)
    if name == "search_jobs"   : return search_jobs(**args)
    if name == "finish_session": return finish_session(**args)
    raise ValueError(f"Unknown tool: {name}")


# ── Tool schema ────────────────────────────────────────────────────────────

AGENT_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_github",
            "description": "Fetch a candidate's real GitHub profile — repos, languages, activity.",
            "parameters": {"type": "object",
                "properties": {"username": {"type": "string"}},
                "required": ["username"]}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_jobs",
            "description": "Search open roles matching the candidate's skills and experience.",
            "parameters": {"type": "object",
                "properties": {
                    "skills"   : {"type": "array", "items": {"type": "string"}},
                    "domain"   : {"type": "string"},
                    "min_years": {"type": "integer"}
                },
                "required": ["skills", "min_years"]}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "finish_session",
            "description": "Call this when the full assessment is complete — GitHub verified, jobs found, recommendation written.",
            "parameters": {"type": "object",
                "properties": {
                    "recommendation": {"type": "string", "description": "2-3 sentence career assessment of the candidate"},
                    "top_jobs"      : {"type": "array",  "items": {"type": "string"},
                                       "description": "Copy the exact job_title and company from search_jobs results. Format: 'job_title at company (location)'. Never invent company names."},
                    "next_steps"    : {"type": "array",  "items": {"type": "string"}, "description": "3 concrete actions for the candidate"}
                },
                "required": ["recommendation", "top_jobs", "next_steps"]}
        }
    },
]

print(f"Agent tools: {[t['function']['name'] for t in AGENT_TOOLS]}")

Agent tools: ['read_github', 'search_jobs', 'finish_session']


---
## Step 3 — The Agent Loop

Same structure as NB03 — but now the agent has a **goal** and an **exit condition**.

```
             ┌─────────────────────────────────┐
             │         AGENT LOOP              │
             │                                 │
  Profile ──►│  LLM decides next action        │
             │       │                         │
             │  tool_calls?                    │
             │   ├── read_github  ──► result   │
             │   ├── search_jobs  ──► result   │
             │   └── finish_session ──► DONE ──┼──► structured output
             │                                 │
             └─────────────────────────────────┘
```

In [9]:
def run_agent(profile: dict) -> dict:
    messages = [
        {"role": "system", "content": AGENT_GOAL},
        {"role": "user",   "content": f"Candidate profile:\n{json.dumps(profile, indent=2)}\n\nBegin the assessment."}
    ]

    for turn in range(1, 10):
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=AGENT_TOOLS
        )
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            # No tool call and no finish_session = unexpected stop
            print(f"Turn {turn}: Agent stopped without finishing.")
            return {}

        for tc in msg.tool_calls:
            args   = json.loads(tc.function.arguments)
            result = execute_tool(tc.function.name, args)

            print(f"  Turn {turn}  →  {tc.function.name}")

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

            # Agent explicitly signals completion via finish_session
            if tc.function.name == "finish_session":
                print(f"\n  Agent completed in {turn} turns.")
                return result

    return {}

print("run_agent() ready.")

run_agent() ready.


---
## Demo — Run the Agent

In [10]:
print("=" * 55)
print("AGENT RUNNING")
print("=" * 55)

output = run_agent(ARJUN)

print()
print("=" * 55)
print("AGENT OUTPUT")
print("=" * 55)
print(f"\nAssessment:\n  {output.get('recommendation')}")
print(f"\nTop Jobs:")
for job in output.get("top_jobs", []):
    print(f"  • {job}")
print(f"\nNext Steps:")
for step in output.get("next_steps", []):
    print(f"  → {step}")

AGENT RUNNING
  Turn 1  →  read_github
  Turn 2  →  search_jobs
  Turn 3  →  finish_session

  Agent completed in 3 turns.

AGENT OUTPUT

Assessment:
  Arjun has a solid background with strong skills in Java, Go, and distributed systems, which are highly relevant in the fintech industry. Based on his experience and preferences for more challenging roles, the recommended positions will allow him to grow further in his career while tackling complex problems.

Top Jobs:
  • Manager, Software Development at Fiserv (Pune - Trion Business Park, India)
  • Software Engineer at Intel (US, Arizona, Phoenix)
  • Senior Software Engineer - Python at PayPal (San Jose, California, United States of America)

Next Steps:
  → Apply to the recommended job positions immediately to increase chances of securing an interview.
  → Consider enhancing his GitHub profile with relevant projects in fintech or distributed systems to showcase his skills.
  → Prepare for interviews by practicing behavior-based ques

---
## The Aha Moment

Nobody told the agent to check GitHub first. Nobody told it when to stop.
It figured that out from its goal.

| What we wrote | What the agent decided |
|---|---|
| The goal (system prompt) | Which tool to call first |
| What tools exist | The order of operations |
| The loop | When the assessment was complete |
| The exit tool (`finish_session`) | What to put in the final output |

**The agent owns the sequence. We own the boundaries.**

---

### What changed across notebooks:

```
NB01  LLM understands unstructured text  →  structured profile
NB02  LLM infers what to ask next        →  complete profile  
NB03  LLM calls tools when it needs data →  grounded answers
NB04  LLM drives the session with a goal →  autonomous assessment  ← you are here
```

---

> **Understand → Infer → Act → *Agent Loop* → Explain**

---
## What's Next?

The agent works. But it's text-in, text-out.
A real career coaching session happens over **voice** — natural, interruptible, human.

How do we turn this agent into a voice conversation?

→ **Notebook 05: Complete Agent**